# Understanding and Implementing Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks by`(Lewis et al., 2020).`

# 

## 1. Introduction and Motivation

Consider that you are building a question answer but then as you proceed all of sudden you hit the wall. You realise that traditional large language models like GPT-3 or BERT store all their knowledge in their parameters (the weights of the neural network) but they static and do not changes with the world. For example if you will asked your personal chatbot which had knowledge till November 2022 that **When did Lionel Messi win the world cup?** then it will answer that **Messi never won the world cup** or **Maximum Messi has reached closes to world cup is to the final in 2014.** It has no of the events that happened after that duration of time.<br>

Now this approach has several other problems also. For example:

1. **Hallucinations:** Models sometimes sound very knowledgeable and sophisticated but they are giving completely trash information about a certain topic. If you have no idea about that field then there is no way to verify those facts while models are confidently making up the facts.

2. **Knowledge Capacity:** Models have limited amoung of knowledge because we have to fit them into parameters but we can't fit billions of parameters which are needed to store knowledge of whole world and also such systems are difficult to scale.

Now by RAG we combine best of both worlds:
1. **Retriver:** helps us find relevant documents from a huge text corpus.
2. **Generator:** conditions on the query and retrieved documents to produce factual answers.

### 1.1 Problem Which RAG Attempts to Solve

Closed-book generators (like vanilla BART or GPT) must store all facts within parameters but they hallucinate on unseen or conflicting queries. Yet most real world tasks such as QA, fact checking, dialogue and summarization require grounding answers in external knowledge. RAG treats doucments as hidden variables that we conclude from observable and measurable variables through a mathematical model. For example consider a query $x$ and model is retrieving the documents $z_1$, $z_2 ... $, and then marginalizes over them while generating the final answer $y$.

## 2. Architecture of RAG

RAG has **three main components**:
```

Input Query (x)
     │
     ▼
┌─────────────────────────┐
│   RETRIEVER p_η(z|x)    │  ← Finds relevant documents
│(Dense Passage Retriev)  │
└──────────┬──────────────┘
           │
           ▼
     Top-K Documents (z)
           │
           ▼
┌─────────────────────────┐
│  GENERATOR p_θ(y|x,z)   │  ← Generates answer
│       (BART)            │
└──────────┬──────────────┘
           │
           ▼
      Answer (y)

```

### 2.1 Understanding the Retriever
DPR stands for Dense Passage Retrieval which is a neural retrieval system introduced by Facebook AI [Karpukhin et al., 2020](https://arxiv.org/abs/2004.04906). It’s deeply connected to how RAG retrieves documents.

DPR represents both queries and documents as continuous vectors using BERT like encoders and retrieves relevant documents by computing vector similarity (typically inner product) between a query vector and millions of document vectors stored in an index. DPR replaced traditional keyword-based search (like TF-IDF, BM25) with a neural, semantic search that works well for open-domain question answering. The retriever hastwo BERT encoders:

- $\text{Query encoder: } q(x) = f_q(x) \in \mathbb{R}^d$
- $\text{Document encoder: } d(z) = f_d(z) \in \mathbb{R}^d$ 


Our query enocder takes input $x$ and converts it into a dense vector representation $q(x)$. Then our document encoder has already been run on the entire knowledge base (e.g., all of Wikipedia, split into 100-word chunks 16). It creates a vector $d(z)$ for every single document $z$. Finally all these document vectors are stored in a massive Document Index18. The model uses **Maximum Inner Product Search (MIPS)** to instantly find the top-K document vectors $d(z)$ that are closest to our query vector $q(x)$

> In layman terms retriever turns our question into a vector and finds the K Wikipedia chunks whose vectors are the most similar.


### 2.2 Inside the Generator
The generator (BART or T5) models the probability of generating the sequence $y = (y_1, \ldots, y_N)$:<br>
$p_\theta(y \mid x, z)
= \prod_{i=1}^{N} p_\theta(y_i \mid x, z, y_{1:i-1})$

Here we are performing following steps:

* **Input:** concatenate `[x; z]` as the encoder input (query + passage).
* **Output:** autoregressive decoding of (y_i) one token at a time.
* **Training:** teacher forcing — the decoder conditions on the ground-truth prefix $y_{1:i-1}$.
* **Inference:** uses beam search or nucleus sampling for decoding.

To explain in short generator is pre-trained sequence-to-sequence model specifically mostly BART-large. The Generator takes both the original input $x$ and the text of a retrieved document $z$ and simply concatenates them and finally auto-regressively generates the final answer $y$.


## 3. The Mathematics of RAG

Let's first establish our notation clearly:

- $x$ : Input query (e.g., "What is the capital of France?")
- $y$ : Output sequence (e.g., "Paris")
- $z$ : Retrieved document
- $y_i$ : The $i$-th token in output sequence
- $N$ : Length of output sequence

RAG models the probability of generating output $y$ given input $x$ as:<br>
$p(y|x) = \sum_{z \in \mathcal{Z}} p(y, z|x)$

Where:
- $\mathcal{Z}$ is the set of all possible documents
- $p(y, z|x)$ is the joint probability of document $z$ and output $y$ given input $x$

By the chain rule of probability:<br>
$p(y, z|x) = p(z|x) \cdot p(y|x, z)$

Therefore $p(y|x) = \sum_{z \in \mathcal{Z}} p(z|x) \cdot p(y|x, z)$

**Interpretation:**
- $p(z|x)$ : **Retriever** - How relevant is document $z$ to query $x$?
- $p(y|x, z)$ : **Generator** - How likely is answer $y$ given query $x$ and document $z$?
- The sum marginalizes over all possible documents

### 3.1 The Top-K Approximation

**Problem:** Summing over ALL documents in Wikipedia (21 million) is computationally infeasible so

**Solution:** Approximate by summing over only the **top-K** most relevant documents:

$$p(y|x) \approx \sum_{z \in \text{top-}K(p(\cdot|x))} p(z|x) \cdot p(y|x, z)$$

Where:
- $\text{top-}K(p(\cdot|x))$ returns the $K$ documents with highest $p(z|x)$.
- Typically $K \in \{5, 10\}$.
- This is the **key approximation** in RAG.


 ## 4. The Interaction Between the Two Blocks
 
 Summing over all documents in Wikipedia which are in millions is computationally impossible therefore the authors approximate this sum by only using the top-K documents found by the retriever.<br>
 The paper proposes two different models for how to do this:

### 4.1 RAG-Sequence Model

RAG-Sequence uses the **same document** for generating the entire output sequence:

$p_{\text{RAG-Seq}}(y|x) \approx \sum_{z \in \text{top-}K(p(\cdot|x))} p_\eta(z|x) \cdot p_\theta(y|x, z)$

Now expanding the generator term:

$p_{\text{RAG-Seq}}(y|x) \approx \sum_{z \in \text{top-}K(p(\cdot|x))} p_\eta(z|x) \prod_{i=1}^{N} p_\theta(y_i|x, z, y_{1:i-1})$

#### Step-by-Step Breakdown

- **Step 1: Retrieve Documents**

For query $x$ retrieve $K$ documents with highest relevance:

$\{z_1, z_2, \ldots, z_K\} = \text{top-}K(p_\eta(\cdot|x))$

- **Step 2: Compute Generation Probability for Each Document**

For each document $z_k$, compute the probability of generating the full sequence:

$p_\theta(y|x, z_k) = \prod_{i=1}^{N} p_\theta(y_i|x, z_k, y_{1:i-1})$

**Step 3: Weight by Retrieval Probability**

Weight each generation probability by how relevant the document is:

$\text{weighted-prob}_k = p_\eta(z_k|x) \cdot p_\theta(y|x, z_k)$

- **Step 4: Marginalize (Sum) Over Documents**

$p_{\text{RAG-Seq}}(y|x) = \sum_{k=1}^{K} p_\eta(z_k|x) \cdot p_\theta(y|x, z_k)$

#### Example Calculation

**Given:**
- Query: "What is the capital of France?"
- Target: "Paris"
- K = 3 documents retrieved

**Step-by-step:**

1. **Retrieval probabilities:**
   $p_\eta(z_1|x) = 0.6, \quad p_\eta(z_2|x) = 0.3, \quad p_\eta(z_3|x) = 0.1$

2. **Generation probabilities:**
   $p_\theta(\text{"Paris"}|x, z_1) = 0.8$
   $p_\theta(\text{"Paris"}|x, z_2) = 0.5$
   $p_\theta(\text{"Paris"}|x, z_3) = 0.2$

3. **Joint probabilities:**
   $0.6 \times 0.8 = 0.48$
   $0.3 \times 0.5 = 0.15$
   $0.1 \times 0.2 = 0.02$

4. **Marginal probability:**
   $p(\text{"Paris"}|x) = 0.48 + 0.15 + 0.02 = 0.65$


### 4.2 RAG-Token Model

RAG-Token can use **different documents for each token**:

$p_{\text{RAG-Token}}(y|x) \approx \prod_{i=1}^{N} \sum_{z \in \text{top-}K(p(\cdot|x))} p_\eta(z|x) \cdot p_\theta(y_i|x, z, y_{1:i-1})$

In RAG token model the sum and product are **swapped** compared to RAG-Sequence

#### Step-by-Step Breakdown

**For each output position $i$:**

- **Step 1: Compute Token Probability for Each Document**

For document $z_k$, compute probability of generating token $y_i$:

$p_\theta(y_i|x, z_k, y_{1:i-1})$

- **Step 2: Weight by Retrieval Probability**

$\text{weighted-prob}_{k,i} = p_\eta(z_k|x) \cdot p_\theta(y_i|x, z_k, y_{1:i-1})$

- **Step 3: Marginalize Over Documents for This Token**

$p(y_i|x, y_{1:i-1}) = \sum_{k=1}^{K} p_\eta(z_k|x) \cdot p_\theta(y_i|x, z_k, y_{1:i-1})$

- **Step 4: Multiply Across All Tokens**

$p_{\text{RAG-Token}}(y|x) = \prod_{i=1}^{N} p(y_i|x, y_{1:i-1})$

#### Mathematical Intuition

- For **RAG-Sequence** pick one document and generate entire sequence with it.

$p(y|x) = \underbrace{\sum_{z}}_{\text{document}} \underbrace{\prod_{i}}_{\text{tokens}} p_\eta(z|x) \cdot p_\theta(y_i|x,z,y_{<i})$

- For **RAG-Token:** for each token marginalize over documents

$p(y|x) = \underbrace{\prod_{i}}_{\text{tokens}} \underbrace{\sum_{z}}_{\text{document}} p_\eta(z|x) \cdot p_\theta(y_i|x,z,y_{<i})$



## 5. Decoding the Answers

Generating an answer at test time (decoding) is tricky because of the marginalization.

- For RAG-Token this is simple. At each step $i$ the model calculates the next-token probability $p'(y_i | ...)$ by summing over the top-K documents. This final and blended probability distribution can be plugged directly into a standard beam search decoder.

- For RAG sequence we have harder problem.

    - **Thorough Decoding:** We have to run a separate beam search for each of your K documents (e.g., 10 beam searches) and this gives us a set of candidate answers. After this we take every candidate and re-score it using the full RAG-Sequence formula (summing its probability across all 10 documents). This approach accurate but very slow.

    - **Fast Decoding:** This is cheaper approximation and we run 10 beam searches. Assume that if an answer $y$ wasn't in the beam for document $z$, its probability $p(y|x,z)$ is just 0. This avoids the slow re-scoring step.

## Implementing the RAG

In [1]:
import faiss
import numpy as np
import torch.nn as nn
import seaborn as sns# for data visualization
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import torch.nn.functional as F
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
from torch.utils.data import Dataset, DataLoader


In [2]:
import torch
torch.manual_seed(42)
np.random.seed(42)

In [3]:
from transformers import (
    BertModel, BertTokenizer,
    BartForConditionalGeneration, BartTokenizer,
    get_linear_schedule_with_warmup
)

from torch.optim import AdamW

### Configuring the RAG Model

In [4]:
@dataclass
#configuring with paper's details
class RAGConfig:
    #model arhitecture parameters
    retriever_model_name: str = 'bert-base-uncased'
    generator_model_name: str = 'facebook/bart-base'
    #settings for retrieval
    n_docs: int = 5# number of documents to retrieve
    max_doc_length: int = 100# max length of each document i.e. 100 word chukns in paper
    retrieval_dim: int = 768# dimension of the retriever embeddings with BERT base hidden size
    
    #generation settings
    max_source_length: int = 512# max length of input sequence to generator
    max_target_length: int = 128# max length of target sequence to generator
    
    #training hyperparameters
    learning_rate: float = 1e-4#from the paper
    warmup_steps: int  = 500#function of warmpup steps is to gradually increase the learning rate at the start of training for better convergence
    gradient_accumulation_steps: int = 1# to simulate larger batch sizes
    max_grad_norm: float = 1.0# to prevent exploding gradients
    batch_size: int = 2# number of samples per batch
    
    #index settings
    use_faiss_gpu: bool = False# whether to use GPU for FAISS index
    index_type: str = 'Flat'# type of FAISS index to use
    
    model_type: str = "rag-sequence"
    

config = RAGConfig()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Configuration: {config}")

Configuration: RAGConfig(retriever_model_name='bert-base-uncased', generator_model_name='facebook/bart-base', n_docs=5, max_doc_length=100, retrieval_dim=768, max_source_length=512, max_target_length=128, learning_rate=0.0001, warmup_steps=500, gradient_accumulation_steps=1, max_grad_norm=1.0, batch_size=2, use_faiss_gpu=False, index_type='Flat', model_type='rag-sequence')


### Building Document Encoder

In [5]:
class DocumentEncoder(nn.Module):
    
    def __init__(self, model_name: str = "bert-base-uncased"):
        super().__init__()
        self.encoder = BertModel.from_pretrained(model_name)
        self.hidden_size = self.encoder.config.hidden_size
    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        #taking input ids and attention mask[batch_size, seq_length] as arguments and returning the embeddings
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.last_hidden_state[:, 0, :]


doc_encoder = DocumentEncoder()
print(f"Hidden size: {doc_encoder.hidden_size}")
print(f"Trainable parameters: {sum(p.numel() for p in doc_encoder.parameters() if p.requires_grad):,}")
print(f"Total parameters: {sum(p.numel() for p in doc_encoder.parameters()):,}")

Hidden size: 768
Trainable parameters: 109,482,240
Total parameters: 109,482,240


### Building Query Enocder

Here we will use BERT_BASE as query encoder and which produces query representation in the form `q(x) = BERT_q(X)`. Its main function is to learn to retrieve documents useful for generation task.

In [6]:
#this will be trained with generator and will recieve gradients from generattion loss
class QueryEncoder(nn.Module):
    
    def __init__(self, model_name: str = "bert-base-uncased"):
        super().__init__()
        self.encoder = BertModel.from_pretrained(model_name)
        self.hidden_size = self.encoder.config.hidden_size
        print("QueryEncoder initialized.")
        
    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)# initializing the BERT model with input ids and attention mask           
        return outputs.last_hidden_state[:, 0, :]# returning the CLS token embeddings

query_encoder = QueryEncoder()# instantiating the QueryEncoder
print(f"Trainable parameters: {sum(p.numel() for p in query_encoder.parameters() if p.requires_grad):,}")# counting the number of trainable parameters
print(f"Total parameters: {sum(p.numel() for p in query_encoder.parameters()):,}")# counting the total number of parameters

QueryEncoder initialized.
Trainable parameters: 109,482,240
Total parameters: 109,482,240


### Building Dense Retriever with FAISS

Here we will implement dense retriever(`p_η(z|x) ∝ exp(d(z)^T q(x))`) as defined in the paper which uses Maximum Inner Product Search(MIPS) with FAISS for efficiency. 

In [7]:
class DenseRetriever:
    
    def __init(self, document_enocder: DocumentEncoder, query_encoder: QueryEncoder, config: RAGConfig):
        self.doc_encoder = document_enocder
        self.query_encoder = query_encoder
        self.config = config
        self.index = None
        self.doc_embeddings = None
        self.documents = []# to store the original documents
        
    def build_index(self, documents: List[str], tokenizer, batch_size: int = 32, device: str = "cuda"):
        self.documents = documents# calling the documents to be indexed
        self.doc_encoder.eval()# setting the document encoder to evaluation mode
        self.doc_encoder.to(device)# moving the document encoder to the specified device
        
        all_embeddings = []# to store all document embeddings
        
        print(f"Building inxed for {len(documents)} documents.")
        with torch.no_grad():# disabling gradient calculation for efficiency
            for i in tqdm(range(0, len(documents), batch_size), desc = "Encoding documents"):
                batch_docs = documents[i:i+batch_size]#getting the current batch of documents
                #tokenizing the batch of documents
                enocded = tokenizer(batch_docs, padding=True, truncation=True, max_length=256, return_tensors="pt")
                
                input_ids = enocded['input_ids'].to(device)
                attention_mask = enocded['attention_mask'].to(device)
                
                #encooding: d(z)
                embeddings = self.doc_encoder(input_ids, attention_mask)# getting the document embeddings
                all_embeddings.append(embeddings.cpu().numpy())# moving embeddings to CPU and converting to numpy array
        
        #concatenating all embeddings
        self.doc_embeddings = np.vstack(all_embeddings).astype('float32')
        
        #building the FAISS index for MIPS
        print("Building FAISS index!")
        d = self.doc_embeddings.shape[1]# dimension of the embeddings
        if self.config.index_type == 'Flat':
            self.index = faiss.IndexFlatIP(d)# using inner product for similarity search
        elif self.config.index_type == 'HNSW':
            self.index = faiss.IndexHNSWFlat(d, 32)# using HNSW index for faster search
        
        #add document vectors to the index
        self.index.add(self.doc_embeddings)
        print(f"FAISS index built with {self.index.ntotal} documents.")
        return self
    
    def retrieve(self, queries: List[str], tokenizer, k:int = None, device: str = "cuda") -> Tuple[torch.Tensor, List[List[str]], torch.Tensor]:
        if k is None:
            k = self.config.n_docs# number of documents to retrieve
            
        if self.index is None:
            raise ValueError("Index not built. Call build_index() first.")

        self.query_encoder.eval()
        self.query_encoder.to(device)

        with torch.no_grad():
            # Tokenize queries
            encoded = tokenizer(
                queries,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt"
            )

            input_ids = encoded["input_ids"].to(device)
            attention_mask = encoded["attention_mask"].to(device)

            # Encode: q(x)
            query_embeddings = self.query_encoder(input_ids, attention_mask)
            query_embeddings_np = query_embeddings.cpu().numpy().astype('float32')

            # MIPS: Find top-k documents by inner product
            scores, indices = self.index.search(query_embeddings_np, k)

            # Convert to probabilities: p_η(z|x) = softmax(d(z)^T q(x))
            scores_tensor = torch.from_numpy(scores).to(device)
            retrieval_probs = F.softmax(scores_tensor, dim=-1)

            # Get document texts
            retrieved_docs = []
            for batch_indices in indices:
                batch_docs = [self.documents[idx] for idx in batch_indices]
                retrieved_docs.append(batch_docs)

            doc_indices = torch.from_numpy(indices).to(device)

        return retrieval_probs, retrieved_docs, doc_indices


print("DenseRetriever class defined.")

DenseRetriever class defined.


### BART Generator

BART is responsible for turning retrieved knowledge (document) and the input query into a final natural language answer. RAG uses BART as this generator because BART is a seq2seq Transformer that’s:
- pretrained to reconstruct masked and corrupted text.
- strong at language generation.
- easily fine-tuned on conditional tasks.

In [8]:
from transformers import BartConfig

class RAGGenerator(nn.Module):
    def __init__(self, model_name: str = "facebook/bart-base"):
        super().__init__()
        self.model_name = model_name
        try:
            # try to load pretrained weights (may fail if credentials/environment block downloads)
            self.model = BartForConditionalGeneration.from_pretrained(model_name)
            print(f"Generator Loaded from pretrained weights: {model_name}")
        except Exception as e:
            # fallback: initialize model from config (random weights) to avoid download/auth errors
            print(f"Warning: failed to load pretrained model '{model_name}' due to: {e}")
            print("Falling back to initializing model from config (random weights).")
            cfg = BartConfig()
            self.model = BartForConditionalGeneration(cfg)

        self.vocab_size = self.model.config.vocab_size
        print(f"Parameters: {sum(p.numel() for p in self.model.parameters() if p.requires_grad):,}")# counting trainable parameters
        
    def forward(self, input_ids: torch.Tensor, attention_mask:torch.Tensor, decoder_input_ids:torch.Tensor,decoder_attention_mask:torch.Tensor = None)->torch.Tensor:
        outputs = self.model(
            input_ids = input_ids,
            attention_mask = attention_mask,
            decoder_input_ids = decoder_input_ids,
            decoder_attention_mask = decoder_attention_mask
        )
        return outputs.logits


generator = RAGGenerator(config.generator_model_name)
print(f"Vocab size: {generator.vocab_size}")

401 Client Error: Unauthorized for url: https://huggingface.co/facebook/bart-base/resolve/main/config.json (Request ID: Root=1-69101069-72ebb063530156604a339009;69b508e7-743b-46f6-8f19-74f73e1fc30a)

Invalid credentials in Authorization header
Falling back to initializing model from config (random weights).
Parameters: 406,291,456
Vocab size: 50265


### Building RAG Sequence Model

RAG Sequence Model uses same document for entire sequence and marginalizes over documents at sequence level.

In [9]:
"""
We will now build the RAG Sequence Model which uses same document for entire sequence and marginalizes over documents at sequence level. It retrieves top-k documents for each query and computes the generation probabilities for each document separately. The final output probabilities are obtained by marginalizing(sum) over the retrieved documents.
"""
class RAGSequenceModel(nn.Module):

    def __init__(self,retriever: DenseRetriever,generator: RAGGenerator,onfig: RAGConfig):
        super().__init__()
        self.retriever = retriever
        self.generator = generator
        self.config = config
        print("RAG-Sequence model initialized.")
    
    #computing loss with marginalization over documents and returns the loss
    def forward(self, query_ids:torch.Tensor, query_mask:torch.Tensor, target_ids:torch.Tensor, target_mask:torch.Tensor, retrieved_doc_ids:torch.Tensor, retrieved_doc_mask:torch.Tensor, retrieval_probs:torch.Tensor) -> torch.Tensor:
        batch_size, n_docs, doc_len = retrieved_doc_ids.size()
        # For each document computong p_θ(y|x,z)
        doc_log_probs = []

        for doc_idx in range(n_docs):
            # Get document
            doc_ids = retrieved_doc_ids[:, doc_idx, :]
            doc_mask = retrieved_doc_mask[:, doc_idx, :]

            # Concatenate [query] [document]
            input_ids = torch.cat([query_ids, doc_ids], dim=1)
            attention_mask = torch.cat([query_mask, doc_mask], dim=1)

            # Prepare decoder input (shift right for teacher forcing)
            decoder_input_ids = self._shift_right(target_ids)

            # Forward: get logits
            logits = self.generator(
                input_ids=input_ids,
                attention_mask=attention_mask,
                decoder_input_ids=decoder_input_ids,
                decoder_attention_mask=target_mask
            )

            # Compute log p_θ(y_i|x,z,y_{<i})
            log_probs = F.log_softmax(logits, dim=-1)

            # Gather target token log probs
            target_log_probs = torch.gather(
                log_probs,
                dim=2,
                index=target_ids.unsqueeze(-1)
            ).squeeze(-1)

            # Mask padding
            target_log_probs = target_log_probs * target_mask

            # Sum over sequence: log p_θ(y|x,z) = Σ_i log p_θ(y_i|x,z,y_{<i})
            sequence_log_prob = target_log_probs.sum(dim=1)
            doc_log_probs.append(sequence_log_prob)

        # Stack: [batch_size, n_docs]
        doc_log_probs = torch.stack(doc_log_probs, dim=1)

        # Add retrieval log probs: log p_η(z|x)
        retrieval_log_probs = torch.log(retrieval_probs + 1e-10)

        # Joint: log p_η(z|x) + log p_θ(y|x,z)
        joint_log_probs = retrieval_log_probs + doc_log_probs

        # Marginalize: log Σ_z exp(log p_η(z|x) + log p_θ(y|x,z))
        marginal_log_prob = torch.logsumexp(joint_log_probs, dim=1)

        # Negative log-likelihood
        loss = -marginal_log_prob.mean()

        return loss

    #shift target ids to the right for decoder input (teacher forcing)
    def _shift_right(self, input_ids: torch.Tensor) -> torch.Tensor:
        decoder_start_token_id = self.generator.model.config.decoder_start_token_id
        shifted_input_ids = input_ids.new_zeros(input_ids.shape)
        shifted_input_ids[:, 1:] = input_ids[:, :-1].clone()
        shifted_input_ids[:, 0] = decoder_start_token_id
        return shifted_input_ids


print("RAG-Sequence model class defined.")

RAG-Sequence model class defined.


### Building RAG Token Model

In [10]:
class RAGTokenModel(nn.Module):
    def __init__(self,retriever: DenseRetriever,
        generator: RAGGenerator,config: RAGConfig):
        super().__init__()
        self.retriever = retriever
        self.generator = generator
        self.config = config
        print("✓ RAG-Token model initialized")

    def forward(self, query_ids: torch.Tensor, query_mask: torch.Tensor,
        target_ids: torch.Tensor, target_mask: torch.Tensor,
        retrieved_doc_ids: torch.Tensor,retrieved_doc_mask: torch.Tensor,
        retrieval_probs: torch.Tensor
    ) -> torch.Tensor:
        batch_size, n_docs, doc_len = retrieved_doc_ids.shape

        # Compute logits for all documents
        all_logits = []

        for doc_idx in range(n_docs):
            doc_ids = retrieved_doc_ids[:, doc_idx, :]
            doc_mask = retrieved_doc_mask[:, doc_idx, :]

            # Concatenate query and document
            input_ids = torch.cat([query_ids, doc_ids], dim=1)
            attention_mask = torch.cat([query_mask, doc_mask], dim=1)

            decoder_input_ids = self._shift_right(target_ids)

            # Forward
            logits = self.generator(
                input_ids=input_ids,
                attention_mask=attention_mask,
                decoder_input_ids=decoder_input_ids,
                decoder_attention_mask=target_mask
            )
            all_logits.append(logits)

        # Stack: [batch_size, n_docs, target_len, vocab_size]
        all_logits = torch.stack(all_logits, dim=1)

        # Log probabilities
        log_probs = F.log_softmax(all_logits, dim=-1)

        # Gather target token log probs: [batch_size, n_docs, target_len]
        target_log_probs = torch.gather(
            log_probs,
            dim=3,
            index=target_ids.unsqueeze(
                1).unsqueeze(-1).expand(-1, n_docs, -1, -1)
        ).squeeze(-1)

        # Add retrieval log probs: [batch_size, n_docs, 1]
        retrieval_log_probs = torch.log(retrieval_probs + 1e-10).unsqueeze(-1)

        # Joint: log p_η(z|x) + log p_θ(y_i|x,z,y_{<i})
        joint_log_probs = retrieval_log_probs + target_log_probs

        # Marginalize over docs for each token: log Σ_z p_η(z|x) p_θ(y_i|...)
        marginal_log_probs = torch.logsumexp(joint_log_probs, dim=1)

        # Mask padding
        marginal_log_probs = marginal_log_probs * target_mask

        # Sum over sequence, mean over batch
        loss = -marginal_log_probs.sum(dim=1).mean()

        return loss

    def _shift_right(self, input_ids: torch.Tensor) -> torch.Tensor:
        decoder_start_token_id = self.generator.model.config.decoder_start_token_id
        shifted_input_ids = input_ids.new_zeros(input_ids.shape)
        shifted_input_ids[:, 1:] = input_ids[:, :-1].clone()
        shifted_input_ids[:, 0] = decoder_start_token_id
        return shifted_input_ids


print("RAG-Token model class defined.")

RAG-Token model class defined.


### Building Dataset and Training

In [11]:
def create_knowledge_base():
    """
    Create a toy knowledge base.
    Paper uses 21M Wikipedia chunks.
    """
    documents = [
        "Paris is the capital and largest city of France, located on the River Seine.",
        "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.",
        "France is a country primarily located in Western Europe.",
        "The Louvre is the world's most-visited museum, located in Paris.",
        "London is the capital and largest city of England and the United Kingdom.",
        "The River Thames flows through southern England, including through London.",
        "Big Ben is the nickname for the Great Bell of the clock at the Palace of Westminster.",
        "Tokyo is the capital of Japan and the most populous city in the world.",
        "Mount Fuji is an active volcano and the highest mountain in Japan.",
        "Japan is an island country in East Asia, located in the northwest Pacific Ocean.",
        "Berlin is the capital and largest city of Germany.",
        "The Berlin Wall was a guarded concrete barrier that divided Berlin from 1961 to 1989.",
        "Germany is a country in Central Europe.",
        "Rome is the capital city of Italy and a special comune.",
        "The Colosseum is an oval amphitheatre in the centre of Rome, Italy.",
        "Madrid is the capital and most populous city of Spain.",
        "The Sagrada Família is a large unfinished Roman Catholic minor basilica in Barcelona.",
        "Moscow is the capital and largest city of Russia.",
        "The Kremlin is a fortified complex in the center of Moscow.",
        "Beijing is the capital of the People's Republic of China."
    ]

    # Question-answer pairs
    qa_pairs = [
        ("What is the capital of France?", "Paris"),
        ("Where is the Eiffel Tower located?", "Paris, France"),
        ("What is the capital of England?", "London"),
        ("What river flows through London?", "River Thames"),
        ("What is the capital of Japan?", "Tokyo"),
        ("What is the highest mountain in Japan?", "Mount Fuji"),
        ("What is the capital of Germany?", "Berlin"),
        ("What is the capital of Italy?", "Rome"),
        ("Where is the Colosseum?", "Rome, Italy"),
        ("What is the capital of Spain?", "Madrid"),
    ]

    return documents, qa_pairs


documents, qa_pairs = create_knowledge_base()
print(f"Knowledge base: {len(documents)} documents")
print(f"Training data: {len(qa_pairs)} QA pairs")
print(f"\nExample document: {documents[0]}")
print(f"Example QA: Q: {qa_pairs[0][0]} and A: {qa_pairs[0][1]}")

Knowledge base: 20 documents
Training data: 10 QA pairs

Example document: Paris is the capital and largest city of France, located on the River Seine.
Example QA: Q: What is the capital of France? and A: Paris


In [12]:
class RAGDataset(Dataset):
    def __init__(self, data, retriever_tokenizer, generator_tokenizer, config):
        self.data = data
        self.retriever_tokenizer = retriever_tokenizer
        self.generator_tokenizer = generator_tokenizer
        self.config = config

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        query, answer = self.data[idx]

        # Tokenize query for retrieval
        query_encoded = self.retriever_tokenizer(
            query,
            max_length=256,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        # Tokenize answer for generation
        answer_encoded = self.generator_tokenizer(
            answer,
            max_length=self.config.max_target_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "query": query,
            "query_ids": query_encoded["input_ids"].squeeze(0),
            "query_mask": query_encoded["attention_mask"].squeeze(0),
            "answer_ids": answer_encoded["input_ids"].squeeze(0),
            "answer_mask": answer_encoded["attention_mask"].squeeze(0)
        }


print("RAGDataset class defined.")

RAGDataset class defined.


### Training Pipeline

In [13]:
print("="*80)
print("Starting RAG Training Pipeline")
print("="*80)

# 1. Initialize tokenizers
print("\n1. Loading tokenizers...")
if 'retriever_tokenizer' in globals():
    print("Using existing retriever_tokenizer")
else:
    retriever_tokenizer = BertTokenizer.from_pretrained(config.retriever_model_name)
    print("Loaded retriever_tokenizer from pretrained.")

# reuse existing generator_tokenizer if present, else try to load BartTokenizer and fallback to retriever_tokenizer
if 'generator_tokenizer' in globals():
    print("Using existing generator_tokenizer")
else:
    try:
        generator_tokenizer = BartTokenizer.from_pretrained(config.generator_model_name)
        print("Loaded generator_tokenizer from pretrained.")
    except Exception as e:
        # fallback: use retriever_tokenizer to avoid network/auth issues
        print(f"Warning: failed to load BartTokenizer ({e}). Falling back to BERT tokenizer for generation tokenization.")
        generator_tokenizer = retriever_tokenizer

print("Tokenizers ready")

# 2. Initialize encoders
print("\n2. Initializing encoders (reuse if already available)...")
# Reuse existing encoders if present to avoid re-initialization and mismatched signatures
if 'doc_encoder' in globals():
    print("Using existing doc_encoder")
else:
    doc_encoder = DocumentEncoder(config.retriever_model_name)
    print("Initialized new doc_encoder")

if 'query_encoder' in globals():
    print("Using existing query_encoder")
else:
    query_encoder = QueryEncoder(config.retriever_model_name)
    print("Initialized new query_encoder")

# 3. Build index
print("\n3. Building FAISS index!!!")
try:
    retriever = DenseRetriever(doc_encoder, query_encoder, config)
    print("Initialized DenseRetriever with (doc_encoder, query_encoder, config)")
except TypeError:
    # fallback: no-arg constructor then attach attributes or call a setter if available
    retriever = DenseRetriever()
    if hasattr(retriever, "set_encoders"):
        # some implementations provide a helper to set encoders/config
        retriever.set_encoders(doc_encoder, query_encoder, config)
        print("Initialized DenseRetriever via set_encoders(...)")
    else:
        # best-effort attribute assignment so build_index can use encoders/config
        retriever.doc_encoder = doc_encoder
        retriever.query_encoder = query_encoder
        retriever.config = config
        print("Initialized DenseRetriever (no-arg) and attached encoders/config manually")

retriever.build_index(documents, retriever_tokenizer, batch_size=4, device=device)
print("Index built.")

# 4. Test retrieval
print("\n4. Testing retrieval...")
test_queries = ["What is the capital of France?"]
probs, docs, indices = retriever.retrieve(test_queries, retriever_tokenizer, k=3, device=device)
print(f"Query: {test_queries[0]}")
for i, (doc, prob) in enumerate(zip(docs[0], probs[0])):
    # ensure scalar float conversion for pretty-printing
    p = float(prob) if isinstance(prob, (torch.Tensor, np.generic)) else float(prob)
    print(f" Doc {i+1} (p={p:.3f}): {doc[:80]}...")

# 5. Initialize generator
print("\n5. Initializing generator!!!")
# reuse existing generator if present, else create (RAGGenerator has safe fallback)
try:
    generator  # if already present
    print("Using existing generator")
except NameError:
    generator = RAGGenerator(config.generator_model_name)
    print("Generator initialized")

# 6. Create RAG model
print(f"\n6. Creating RAG model ({config.model_type})...")
if 'sequence' in config.model_type.lower():
    rag_model = RAGSequenceModel(retriever, generator, config)
else:
    rag_model = RAGTokenModel(retriever, generator, config)
rag_model.to(device)
print("RAG model created.")

# 7. Prepare data
print("\n7. Preparing datasets!!!")
train_data = qa_pairs[:8]
val_data = qa_pairs[8:]
train_dataset = RAGDataset(train_data, retriever_tokenizer, generator_tokenizer, config)
val_dataset = RAGDataset(val_data, retriever_tokenizer, generator_tokenizer, config)
print(f"   Train: {len(train_dataset)} samples")
print(f"   Val: {len(val_dataset)} samples")

print("\n" + "="*80)
print("Setup complete! Ready for training.")
print("="*80)

Starting RAG Training Pipeline

1. Loading tokenizers...
Loaded retriever_tokenizer from pretrained.
401 Client Error: Unauthorized for url: https://huggingface.co/facebook/bart-base/resolve/main/tokenizer_config.json (Request ID: Root=1-69101073-770ab8c557e96f5f46736f51;f756c988-2078-4bda-96a1-e0bb7081f2e6)

Invalid credentials in Authorization header). Falling back to BERT tokenizer for generation tokenization.
Tokenizers ready

2. Initializing encoders (reuse if already available)...
Using existing doc_encoder
Using existing query_encoder

3. Building FAISS index!!!
Initialized DenseRetriever (no-arg) and attached encoders/config manually
Building inxed for 20 documents.


Encoding documents:   0%|          | 0/5 [00:00<?, ?it/s]

Building FAISS index!
FAISS index built with 20 documents.
Index built.

4. Testing retrieval...
Query: What is the capital of France?
 Doc 1 (p=0.554): Germany is a country in Central Europe....
 Doc 2 (p=0.446): France is a country primarily located in Western Europe....
 Doc 3 (p=0.000): Berlin is the capital and largest city of Germany....

5. Initializing generator!!!
Using existing generator

6. Creating RAG model (rag-sequence)...
RAG-Sequence model initialized.
RAG model created.

7. Preparing datasets!!!
   Train: 8 samples
   Val: 2 samples

Setup complete! Ready for training.


In [14]:
# Custom collate function for on-the-fly retrieval
def collate_fn_rag(batch):
    queries = [item["query"] for item in batch]
    query_ids = torch.stack([item["query_ids"] for item in batch])
    query_mask = torch.stack([item["query_mask"] for item in batch])
    answer_ids = torch.stack([item["answer_ids"] for item in batch])
    answer_mask = torch.stack([item["answer_mask"] for item in batch])

    # Perform retrieval
    retrieval_probs, retrieved_docs, doc_indices = retriever.retrieve(
        queries, retriever_tokenizer, k=config.n_docs, device=device
    )

    # Tokenize retrieved documents
    all_doc_ids = []
    all_doc_masks = []

    for batch_docs in retrieved_docs:
        doc_encoded = generator_tokenizer(
            batch_docs,
            max_length=config.max_source_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        all_doc_ids.append(doc_encoded["input_ids"])
        all_doc_masks.append(doc_encoded["attention_mask"])

    retrieved_doc_ids = torch.stack(all_doc_ids)
    retrieved_doc_mask = torch.stack(all_doc_masks)

    return {
        "query_ids": query_ids.to(device),
        "query_mask": query_mask.to(device),
        "answer_ids": answer_ids.to(device),
        "answer_mask": answer_mask.to(device),
        "retrieved_doc_ids": retrieved_doc_ids.to(device),
        "retrieved_doc_mask": retrieved_doc_mask.to(device),
        "retrieval_probs": retrieval_probs
    }


# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    collate_fn=collate_fn_rag
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    collate_fn=collate_fn_rag
)

print(f"DataLoaders created!!")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

DataLoaders created!!
Train batches: 4
Val batches: 1


In [15]:
# Training setup
print("\nSetting up training...")

# Optimizer for only query encoder and generator (document encoder frozen)
optimizer = AdamW([
    {"params": rag_model.retriever.query_encoder.parameters(),
     "lr": config.learning_rate},
    {"params": rag_model.generator.parameters(), "lr": config.learning_rate}
])

# Learning rate scheduler
num_epochs = 3
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer,
    num_warmup_steps=config.warmup_steps,num_training_steps=total_steps)

print(f"Optimizer configured")
print(f"Learning rate: {config.learning_rate}")
print(f"Total steps: {total_steps}")
print(f"Warmup steps: {config.warmup_steps}")


Setting up training...
Optimizer configured
Learning rate: 0.0001
Total steps: 12
Warmup steps: 500


In [ ]:
# Training loop
print("\n" + "="*80)
print("Starting Training")
print("="*80)

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    print("-" * 80)

    # Training
    rag_model.train()
    epoch_train_loss = 0.0

    progress_bar = tqdm(train_loader, desc="Training")
    for batch_idx, batch in enumerate(progress_bar):
        optimizer.zero_grad()

        # Forward pass
        loss = rag_model(
            query_ids=batch["query_ids"],
            query_mask=batch["query_mask"],
            target_ids=batch["answer_ids"],
            target_mask=batch["answer_mask"],
            retrieved_doc_ids=batch["retrieved_doc_ids"],
            retrieved_doc_mask=batch["retrieved_doc_mask"],
            retrieval_probs=batch["retrieval_probs"]
        )

        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            rag_model.parameters(), config.max_grad_norm)
        optimizer.step()
        scheduler.step()

        epoch_train_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_train_loss = epoch_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # Validation
    rag_model.eval()
    epoch_val_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            loss = rag_model(
                query_ids=batch["query_ids"],
                query_mask=batch["query_mask"],
                target_ids=batch["answer_ids"],
                target_mask=batch["answer_mask"],
                retrieved_doc_ids=batch["retrieved_doc_ids"],
                retrieved_doc_mask=batch["retrieved_doc_mask"],
                retrieval_probs=batch["retrieval_probs"]
            )
            epoch_val_loss += loss.item()

    avg_val_loss = epoch_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    print(f"\nEpoch {epoch + 1} Summary:")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss: {avg_val_loss:.4f}")

print("\n" + "="*80)
print("Training Complete!")
print("="*80)